# Flat evaluation reporting demo

Demonstrates **Layer S** (structural row pairing), **Layer 1** (applicability), and **Layer 2** (value matching) on a collated checklist document.

**Model schema** (`schema.json`) — what one model call returns:
- Root key `outputs[]`: panel rows (`label`, `is_micrograph`, `caption_snippet`)

**Evaluation gold/pred** — collation wrappers around repeated model calls:
- `papers[]` → `figures[]` (structural, positional join)
- each figure embeds `outputs[]` (predictive, Hungarian alignment on `label`)

The notebook uses `FlatEvaluator` (phase 5 + 5.1) rather than wiring modules by hand.

Toy errors in pred:
- Figure 1 — perfect
- Figure 2 — reordered outputs; wrong `2A.is_micrograph`; `figure_label` typo
- Figure 3 — 3B missing; spurious 3C

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from soda_mmqc.core.collation import discover_collation_layout
from soda_mmqc.core.evaluation import FlatEvaluator, format_ancestor_context


In [ ]:
def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "soda_mmqc").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    return here


ROOT = find_repo_root()
DEMO_DIR = ROOT / "notebooks/fixtures/flat-eval-demo"

evaluator = FlatEvaluator.from_paths(
    str(DEMO_DIR / "schema.json"),
    str(DEMO_DIR / "manifest.json"),
)
gold = json.loads((DEMO_DIR / "gold.json").read_text())
pred = json.loads((DEMO_DIR / "pred.json").read_text())

layout = discover_collation_layout(evaluator.schema, gold, pred)
result = evaluator.evaluate(gold, pred)
manifest = evaluator.manifest
PREDICTIVE_LIST_KEY = layout.predictive_lists[0].by_list_key

print(f"Checklist: {manifest.checklist}")
print(f"Embedding prefix: {'.'.join(layout.embedding_prefix)}")
print(f"Predictive list: {PREDICTIVE_LIST_KEY}")
print(f"Profiled leaves: {manifest.profiled_leaf_properties()}")

## Fixture overview

| Figure | Gold | Pred (errors) |
|--------|------|---------------|
| 1 | Perfect match | Perfect match |
| 2 | Outputs 2A/2B | Reordered; `2A.is_micrograph` wrong; `figure_label` typo |
| 3 | Outputs 3A/3B | 3A correct; 3B missing; spurious 3C |

In [ ]:
pd.set_option("display.max_colwidth", 60)

display(pd.DataFrame({"schema (model call)": [json.dumps(evaluator.schema, indent=2)]}))
display(pd.DataFrame({"manifest": [json.dumps(json.loads((DEMO_DIR / "manifest.json").read_text()), indent=2)]}))
display(pd.DataFrame({"gold": [json.dumps(gold, indent=2)], "pred": [json.dumps(pred, indent=2)]}))

In [ ]:
LAYER_S_ORDER = ("correct_row", "missing_row", "spurious_row")
LAYER1_ORDER = (
    "correct_NA",
    "correct_applicable",
    "withheld_applicable",
    "spurious_applicable",
)
LAYER2_ORDER = ("TP", "TN", "FP", "FN", "match", "mismatch")

OUTCOME_COLORS = {
    "correct_row": "#2ca02c",
    "missing_row": "#ff7f0e",
    "spurious_row": "#d62728",
    "correct_NA": "#aec7e8",
    "correct_applicable": "#2ca02c",
    "withheld_applicable": "#ff7f0e",
    "spurious_applicable": "#d62728",
    "TP": "#2ca02c",
    "TN": "#98df8a",
    "FP": "#d62728",
    "FN": "#ff7f0e",
    "match": "#2ca02c",
    "mismatch": "#d62728",
}


def counts_to_frame(counts, order: tuple[str, ...], label: str) -> pd.DataFrame:
    return pd.DataFrame({label: list(order), "count": [counts.get(key, 0) for key in order]})


def plot_outcome_bar(counts, order: tuple[str, ...], *, title: str, x_label: str):
    labels = list(order)
    values = [counts.get(label, 0) for label in labels]
    colors = [OUTCOME_COLORS.get(label, "#7f7f7f") for label in labels]
    fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))
    fig.update_layout(title=title, xaxis_title=x_label, yaxis_title="count", yaxis=dict(rangemode="tozero"))
    return fig


instances_df = pd.DataFrame([instance.to_dict() for instance in result.instances]).rename(
    columns={"path": "instance_path", "exp_value": "gold", "pred_value": "pred"}
)
layer_s_payload = result.by_list[PREDICTIVE_LIST_KEY]
layer_s_counts = layer_s_payload["row_counts"]
layer1_counts = result.aggregate_layer1_counts()
layer2_counts = result.aggregate_layer2_counts()

print("Aggregated instance counts:")
display(instances_df.groupby(["layer1", "layer2"], dropna=False).size().reset_index(name="count"))


## Layer S — predictive list row reporting (`by_list`)

Row-slot outcomes for `papers[].figures[].outputs[]` (Hungarian pairing on `label`).
Structural lists (`papers[]`, `figures[]`) use positional join and do not appear in `by_list`.

Missing and spurious rows include paper / figure / panel labels so you can see the culprit immediately.

In [ ]:
layer_s_summary = counts_to_frame(layer_s_counts, LAYER_S_ORDER, "structural")
display(layer_s_summary)

fig_s = plot_outcome_bar(
    layer_s_counts,
    LAYER_S_ORDER,
    title=f"Layer S — {PREDICTIVE_LIST_KEY} row outcomes (all figures)",
    x_label="structural outcome",
)
fig_s.show()

issues = result.layer_s_issues(PREDICTIVE_LIST_KEY)


def issue_table(issue_rows, *, side: str, alignment_col: str) -> pd.DataFrame:
    ancestor_col = f"ancestor_{side}"
    rows = []
    for row in issue_rows:
        rows.append(
            {
                "location": format_ancestor_context(row[ancestor_col]),
                alignment_col: row[f"{side}_alignment"],
                "context_path": row["context_path"],
            }
        )
    return pd.DataFrame(rows)


missing_df = issue_table(issues["missing"], side="gold", alignment_col="gold_row")
spurious_df = issue_table(issues["spurious"], side="pred", alignment_col="pred_row")

print("Missing rows (gold row with no pred match)")
display(missing_df if not missing_df.empty else pd.DataFrame(columns=["location", "gold_row", "context_path"]))

print("Spurious rows (pred row with no gold match)")
display(spurious_df if not spurious_df.empty else pd.DataFrame(columns=["location", "pred_row", "context_path"]))

## Layer 1 — applicability reporting

Was the field answered when it should (or should not) have been?

In [ ]:
layer1_summary = counts_to_frame(layer1_counts, LAYER1_ORDER, "layer1")
display(layer1_summary)
display(instances_df.sort_values("instance_path")[["instance_path", "gold", "pred", "layer1"]])

fig_l1 = plot_outcome_bar(
    layer1_counts,
    LAYER1_ORDER,
    title="Layer 1 — applicability outcomes",
    x_label="layer 1 outcome",
)
fig_l1.show()

## Layer 2 — value matching reporting

Only instances with `layer1 = correct_applicable` receive a layer 2 label.
Binary polarity fields use TP/FP/FN/TN; graded strings use match/mismatch.

In [ ]:
layer2_eligible = instances_df[instances_df["layer1"] == "correct_applicable"].copy()
layer2_summary = counts_to_frame(layer2_counts, LAYER2_ORDER, "layer2")
display(layer2_summary)
display(
    layer2_eligible.sort_values("instance_path")[
        ["instance_path", "gold", "pred", "score", "layer2"]
    ]
)

fig_l2 = plot_outcome_bar(
    layer2_counts,
    LAYER2_ORDER,
    title="Layer 2 — matching outcomes (correct_applicable only)",
    x_label="layer 2 outcome",
)
fig_l2.show()

In [ ]:
layer2_by_field = (
    layer2_eligible.assign(
        field=layer2_eligible["instance_path"].str.rsplit(".", n=1).str[-1]
    )
    .groupby(["field", "layer2"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=LAYER2_ORDER, fill_value=0)
)
display(layer2_by_field)

melted = layer2_by_field.reset_index().melt(
    id_vars="field", var_name="layer2", value_name="count"
)
melted = melted[melted["count"] > 0]
fig_l2_field = px.bar(
    melted,
    x="field",
    y="count",
    color="layer2",
    barmode="stack",
    category_orders={"layer2": list(LAYER2_ORDER)},
    color_discrete_map=OUTCOME_COLORS,
    title="Layer 2 outcomes by leaf field",
)
fig_l2_field.show()

## Combined dashboard

Side-by-side summary of all three reporting layers.

In [ ]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("Layer S", "Layer 1", "Layer 2"),
)

for col, (counts, order) in enumerate(
    [
        (layer_s_counts, LAYER_S_ORDER),
        (layer1_counts, LAYER1_ORDER),
        (layer2_counts, LAYER2_ORDER),
    ],
    start=1,
):
    labels = list(order)
    values = [counts.get(label, 0) for label in labels]
    colors = [OUTCOME_COLORS.get(label, "#7f7f7f") for label in labels]
    fig.add_trace(
        go.Bar(x=labels, y=values, marker_color=colors, showlegend=False),
        row=1,
        col=col,
    )

fig.update_layout(
    title_text="Flat evaluation reporting — collated papers / figures / outputs",
    height=420,
    yaxis=dict(rangemode="tozero"),
    yaxis2=dict(rangemode="tozero"),
    yaxis3=dict(rangemode="tozero"),
)
fig.show()